# Tahap 2: Data Cleaning

Notebook ini merepresentasikan blok mandiri **Data Cleaning** pada diagram metodologi penelitian.

### Deskripsi Tahapan:
1. Menghapus tautan/URL (`https://t.co/...`, `http://...`, `www...`).
2. Menghapus *mention* (`@username`) dan *hashtag* (`#tagar`) atau karakter simbol tagar.
3. Menghapus emoji, emotikon, karakter non-ASCII, angka, dan tanda baca.
4. Merapikan spasi ganda dan karakter baris baru (`\n`, `\r`, `\t`).
5. Menghapus dokumen yang kosong atau terlalu pendek (< 3 kata) setelah dibersihkan (contoh: tweet yang hanya berisi `'Baca: https://t.co/...'`) serta menghapus duplikasi teks bersih.

In [ ]:
import os
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 200)

## 2.1 Memuat Dataset dari Tahap Pengumpulan Data

In [ ]:
input_file = '01_raw_verified.csv' if os.path.exists('01_raw_verified.csv') else 'karhutla_dataset.csv'
df = pd.read_csv(input_file)
print(f'Jumlah data awal sebelum Data Cleaning: {len(df)} baris')
df.head(5)

## 2.2 Fungsi Pembersihan Data (*Data Cleaning*)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    
    # 1. Hapus URL / Link (http, https, www, t.co)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    
    # 2. Hapus mention (@username) dan RT
    text = re.sub(r'\bRT\b', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)
    
    # 3. Hapus hashtag (#hashtag)
    text = re.sub(r'#[A-Za-z0-9_]+', ' ', text)
    
    # 4. Hapus karakter HTML entities (seperti &amp;, &gt;, &lt;)
    text = re.sub(r'&\w+;', ' ', text)
    
    # 5. Hapus emoji, simbol non-ASCII, angka, dan tanda baca (sisakan huruf alfabet dan spasi)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 6. Hapus spasi berlebih dan newline
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['text_cleaned'] = df['text'].apply(clean_text)
df[['text', 'text_cleaned']].head(10)

## 2.3 Filtering Baris Kosong, Terlalu Pendek, dan Duplikat Pasca-Cleaning

In [ ]:
initial_count = len(df)

# Hitung jumlah kata setelah dibersihkan
df['clean_word_count'] = df['text_cleaned'].apply(lambda x: len(x.split()))

# Hapus baris kosong atau yang hanya menyisakan kurang dari 3 kata (misal hanya 'Baca di')
df_cleaned = df[df['clean_word_count'] >= 3].copy()
after_empty_removal = len(df_cleaned)

# Hapus duplikat berdasarkan teks yang sudah dibersihkan (case-insensitive)
df_cleaned['temp_lower'] = df_cleaned['text_cleaned'].str.lower()
df_cleaned = df_cleaned.drop_duplicates(subset=['temp_lower']).drop(columns=['temp_lower']).reset_index(drop=True)
final_count = len(df_cleaned)

print(f'Jumlah data awal                       : {initial_count}')
print(f'Dihapus karena kosong / < 3 kata       : {initial_count - after_empty_removal}')
print(f'Dihapus karena duplikat pasca-cleaning : {after_empty_removal - final_count}')
print(f'Jumlah data bersih akhir               : {final_count}')

## 2.4 Menyimpan Hasil Data Cleaning
File `02_cleaned_data.csv` akan digunakan sebagai input pada notebook **`03_Text_Preprocessing.ipynb`**.

In [ ]:
df_cleaned[['id', 'text', 'text_cleaned']].to_csv('02_cleaned_data.csv', index=False)
print('Hasil Data Cleaning berhasil disimpan ke: 02_cleaned_data.csv')